In [1]:
import os
import json
from tavily import TavilyClient
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv()


True

In [2]:
llm = OpenAI()

In [3]:
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [4]:
embed = llm.embeddings.create


In [5]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            k=5
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=k,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            
            return docs

       


In [6]:
openai_retreiver = retriever(embed, index)

In [7]:
system_prompt = """You are an AI assistant specialized in generating accurate, well-supported responses using both initial and corrective information. Your responses must:

1. Integrate information from multiple context sources coherently
2. Address identified gaps and issues explicitly
3. Resolve any conflicts between different context sources
4. Maintain clear attribution for information sources
5. Acknowledge uncertainties when present

When generating responses:
- Balance information from all provided contexts
- Ensure factual accuracy with evidence
- Structure information logically
- Address all evaluation feedback points
- Stay aligned with the original query intent
- Use clear citations for specific claims"""

In [54]:
context = openai_retreiver.get_data("how much do I need to pay in total to open a hospital with 200 beds?")

In [55]:
class generator:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"""Please generate a comprehensive response to this query:

Original Query: {query}

Using the following information:

Initial Context:
{formatted_context}"""
        }
    ]
)
        evaluation_response = json.loads(self._evaluate(query,context,response.choices[0].message.content))
        regeneration, reretrieval,query_new = evaluation_response["requires_regeneration"],evaluation_response["requires_retrieval"],evaluation_response["query"]
        if reretrieval or regeneration:
            print(query_new)
            context_new = self.retriever.get_data(query =query_new)
            formatted_context_new = "\n".join([str(doc) for doc in context_new])
            response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": f"""Please generate a comprehensive response to this query:

Original Query: {query_new}

Using the following information:

Initial Context:
{formatted_context_new}"""
        }
    ]
)
        return response.choices[0].message.content
    


    def _evaluate(self,query, context, generation):
        schema= {
    "type": "json_schema",
    "json_schema": {
      "name": "llm_evaluation",
      "strict": True,
      "schema": {
        "type": "object",
        "properties": {
          "requires_retrieval": {
            "type": "boolean",
            "description": "Indicates if additional context retrieval is needed."
          },
          "query": {
            "type": "string",
            "description": "A query to get extra context chunks for missing information."
          },
          "requires_regeneration": {
            "type": "boolean",
            "description": "Indicates if the response needs to be regenerated."
          }
        },
        "required": [
          "requires_retrieval",
          "query",
          "requires_regeneration"
        ],
        "additionalProperties": False
      }
    }
  }
        response = llm.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {
      "role": "system",
      "content": [
        {
          "type": "text",
          "text": "You are an AI evaluator responsible for determining the quality of a generated response\nin a Retrieval-Augmented Generation (RAG) system. Your role is to evaluate the\naccuracy, relevance, and completeness of the response based on the provided query and the context chunks.\n\n### Task:\nEvaluate whether the generated response sufficiently answers the query using the retrieved context\nand determine if:\n1. Additional retrieval is required to provide a better answer.\n2. The response should be regenerated based on its quality.\n\n### Input:\n1. **Query**: The user's input or question.\n2. **Context Chunks**: A list of text chunks retrieved from a knowledge base, intended to assist in answering the query.\n3. **Generated Response**: The system's response to the query.\n\n### Output:\nProvide your evaluation as a structured JSON object in the following format:\n- \"query\": Restate the input query.\n- \"retrieved_context\": A summary of the retrieved context chunks or the key points.\n- \"generated_response\": Restate the generated response.\n- \"verdict\":\n  - \"_requires_retrieval\": A boolean indicating whether additional context retrieval is necessary.\n  - \"_retrieval_query\": If additional retrieval is needed, provide a new query or refinement to guide the retrieval process. Otherwise, this field should be null.\n  - \"_requires_regeneration\": A boolean indicating whether the generated response needs to be regenerated.\n- \"overall_feedback\": A concise explanation of your reasoning, including what is missing or why the response is sufficient.\n\n### Example Input:\nQuery: \"What are the health benefits of regular exercise?\"\nContext Chunks:\n1. \"Regular exercise improves cardiovascular health by strengthening the heart and improving blood circulation.\"\n2. \"It helps with weight management by burning calories and building muscle.\"\n3. \"Exercise reduces stress and anxiety by releasing endorphins.\"\n4. \"It also improves sleep quality and increases energy levels.\"\nGenerated Response: \"Regular exercise strengthens the heart and helps manage weight.\"\n\n### Example Output:\n{\n    \"query\": \"What are the health benefits of regular exercise?\",\n    \"retrieved_context\": [\n        \"Regular exercise improves cardiovascular health by strengthening the heart and improving blood circulation.\",\n        \"It helps with weight management by burning calories and building muscle.\",\n        \"Exercise reduces stress and anxiety by releasing endorphins.\",\n        \"It also improves sleep quality and increases energy levels.\"\n    ],\n    \"generated_response\": \"Regular exercise strengthens the heart and helps manage weight.\",\n    \"verdict\": {\n        \"_requires_retrieval\": True,\n        \"_retrieval_query\": \"Provide additional information on the mental health and sleep benefits of regular exercise.\",\n        \"_requires_regeneration\": True\n    },\n    \"overall_feedback\": \"The response is incomplete as it omits critical details about reducing stress, improving sleep quality, and increasing energy levels. Additional retrieval focused on mental health and sleep benefits is recommended. The context provided so far is insufficient, and the response needs regeneration based on more comprehensive retrieval.\"\n}\n\n### Instructions:\nEvaluate the provided query, context chunks, and generated response. If additional retrieval is required, specify a refined query for retrieval in the verdict object. Provide your output in the same structured JSON format as shown above."
        }
      ],
    },
    {
       "role": "user",
      "content": [
          {    
          "type":"text",
          "text": f"query: {query}\nresponse:{generation}\nchunks {context} "
          }
      ] 
    }
  ],
  response_format=schema,
  temperature=0,
  max_completion_tokens=250,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
)
        return response.choices[0].message.content



In [56]:
openai_generator = generator(llm,openai_retreiver)

In [57]:
context

['center, emergency center, home medical center: AED 2000General clinic, specialist clinic, medical center, medical diagnosis center, rehabilitation center: AED 1000Final License fees:General clinic, rehabilitation center, home health center: AED 5,000Specialist clinic, medical diagnosis center, emergency center: AED 6,000Multi-specialty clinic: AED 12000 to AED 18000 maximumFertility center, convalescent home, one-day surgeries hospital: AED 20000Hospital (1 to 50 beds): AED 20000Hospital (50 to 100 beds): AED',
 'center, emergency center, home medical center: AED 1,000General clinic, specialist clinic, medical center, medical diagnosis center, rehabilitation center: AED 500Final license fee:General clinic, rehabilitation center, home health center: AED 5,000Specialist clinic, medical diagnosis center, emergency center: AED 6,000Multi-specialty clinic: AED 12,000 to 18,000 maximumFertility center, convalescent home, one-day surgery hospital: AED 20,000Hospital (1 to 50 beds): AED 20,0

In [58]:

eval = openai_generator.generate("how much do I need to pay in total to open a hospital with 200 beds?", context)

What is the total cost to open a hospital with 200 beds, including all necessary expenses?


In [59]:
eval

'To estimate the total cost of opening a hospital with 200 beds, we must consider several expenses, including licensing fees, construction and equipment costs, staffing, and operational expenses. Based on the initial context, the licensing fees for a hospital with more than 100 beds (which includes a 200-bed capacity) is AED 40,000.\n\n**1. Licensing Fees:**\n- The fee for licensing a hospital with more than 100 beds is AED 40,000. This is a standard payment required to operate legally under health regulations, managed via the MOHAP (Ministry of Health and Prevention) in the UAE.\n\n**2. Construction Costs:**\n- Building a hospital involves significant construction expenditures, which can vary greatly depending on location, materials, design, and local construction costs. In the UAE, estimates often run between AED 2 million to AED 5 million per hospital bed, resulting in a potential total of AED 400 million to AED 1 billion for a 200-bed hospital.\n\n**3. Equipment and Technology:**\n

In [ ]:
# ## Test
# query="what is the the service i use to register as new practicing doctor in the UAE "
# res = openai_retreiver.get_data(query)
# print(res)
# print(type(res))